In [0]:
WITH params AS (
  SELECT
    date_trunc('MONTH', current_date()) AS this_month_start,
    date_trunc('YEAR',  current_date()) AS year_start
),
bounds AS (
  SELECT
    (SELECT year_start       FROM params) AS cy_start_incl,
    (SELECT this_month_start FROM params) AS cy_end_excl,

    add_months((SELECT year_start       FROM params), -12) AS py_start_incl,
    add_months((SELECT this_month_start FROM params), -12) AS py_end_excl
),

/* -----------------------------
   Sessions data
   ----------------------------- */
sessions_q AS (
  
  SELECT
    dl.region,
    CASE
      WHEN to_date(sp.date) >= (SELECT cy_start_incl FROM bounds)
       AND to_date(sp.date) <  (SELECT cy_end_excl  FROM bounds) THEN 'CY'
      WHEN to_date(sp.date) >= (SELECT py_start_incl FROM bounds)
       AND to_date(sp.date) <  (SELECT py_end_excl  FROM bounds) THEN 'PY'
      ELSE NULL
    END AS period,

    date_trunc('QUARTER', to_date(sp.date)) AS quarter_start,

    COUNT(DISTINCT sp.session_id) AS sessions,
    COUNT(DISTINCT sp.visitor_id) AS visitors,
    COUNT(DISTINCT IF(sp.is_bounce, sp.session_id, NULL)) AS bounced_sessions,
    COUNT(DISTINCT IF(sp.adp_views > 0, sp.visitor_id, NULL)) AS quoted_visitors
  FROM production.MARKETPLACE_REPORTS.AGG_SESSION_PERFORMANCE sp
  left join production.dwh.dim_location dl on dl.country_id = sp.ip_geo_country_id
  WHERE
    (
      to_date(sp.date) >= (SELECT cy_start_incl FROM bounds)
      AND to_date(sp.date) <  (SELECT cy_end_excl  FROM bounds)
    )
    OR
    (
      to_date(sp.date) >= (SELECT py_start_incl FROM bounds)
      AND to_date(sp.date) <  (SELECT py_end_excl  FROM bounds)
    )
  GROUP BY 1,2,3
),
sessions_q_final AS (
  SELECT
    region,
    period,
    quarter_start,
    sessions,
    visitors,
    (bounced_sessions / NULLIF(sessions, 0)) AS bounce_rate,
    (quoted_visitors  / NULLIF(visitors, 0)) AS quoter_rate
  FROM sessions_q
  WHERE period IS NOT NULL
),

/* -----------------------------
   Bookings (single pass)
   ----------------------------- */
bookings_q AS (
  SELECT
    region,
    CASE
      WHEN to_date(fb.date_of_checkout) >= (SELECT cy_start_incl FROM bounds)
       AND to_date(fb.date_of_checkout) <  (SELECT cy_end_excl  FROM bounds) THEN 'CY'
      WHEN to_date(fb.date_of_checkout) >= (SELECT py_start_incl FROM bounds)
       AND to_date(fb.date_of_checkout) <  (SELECT py_end_excl  FROM bounds) THEN 'PY'
      ELSE NULL
    END AS period,

    date_trunc('QUARTER', to_date(fb.date_of_checkout)) AS quarter_start,

    -- keep your status logic (=1). Change to IN (1,2) if that's your "active" definition.
    COUNT(DISTINCT CASE WHEN coalesce(fb.status_id, -1) = 1 THEN fb.booking_id END)        AS bookings,
    COUNT(DISTINCT CASE WHEN coalesce(fb.status_id, -1) = 1 THEN fb.shopping_cart_id END) AS transactions,

    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.gmv, 0) ELSE 0 END)               AS gmv_eur,
    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.nr, 0) ELSE 0 END)                AS nr_eur,
    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.nr_supplier_share, 0) ELSE 0 END) AS nr_bt_eur,
    SUM(CASE WHEN coalesce(fb.status_id, -1) = 1 THEN coalesce(fb.net_tax_eur, 0) ELSE 0 END)       AS net_tax_eur
  FROM production.dwh.fact_booking_v2 fb
  left join production.dwh.dim_location dl on dl.location_id = fb.location_id
  WHERE fb.date_of_checkout IS NOT NULL
    AND (
      (
        to_date(fb.date_of_checkout) >= (SELECT cy_start_incl FROM bounds)
        AND to_date(fb.date_of_checkout) <  (SELECT cy_end_excl  FROM bounds)
      )
      OR
      (
        to_date(fb.date_of_checkout) >= (SELECT py_start_incl FROM bounds)
        AND to_date(fb.date_of_checkout) <  (SELECT py_end_excl  FROM bounds)
      )
    )
  GROUP BY 1,2,3
),
bookings_q_final AS (
  SELECT *
  FROM bookings_q
  WHERE period IS NOT NULL
),

/* -----------------------------
   Combine sessions + bookings per (period, quarter)
   ----------------------------- */
q_all AS (
  SELECT
    coalesce(s.region, b.region) AS region,
    coalesce(s.period, b.period) AS period,
    coalesce(s.quarter_start, b.quarter_start) AS quarter_start,

    s.sessions,
    s.visitors,
    s.bounce_rate,
    s.quoter_rate,

    b.bookings,
    b.transactions,
    b.gmv_eur,
    b.nr_eur,
    b.nr_bt_eur,
    b.net_tax_eur,

    CASE WHEN b.gmv_eur = 0 THEN NULL ELSE b.nr_eur    / b.gmv_eur END AS take_rate_nr,
    CASE WHEN b.gmv_eur = 0 THEN NULL ELSE b.nr_bt_eur / b.gmv_eur END AS take_rate_nr_bt
  FROM sessions_q_final s
  LEFT JOIN bookings_q_final b
    ON b.period = s.period
   AND b.quarter_start = s.quarter_start
   AND b.region = s.region
),

cy AS (SELECT * FROM q_all WHERE period = 'CY'),
py AS (SELECT * FROM q_all WHERE period = 'PY'),

/* -----------------------------
   Final wide-by-quarter rows (one row per CY quarter)
   ----------------------------- */
final AS (
  SELECT
    cy.region,
    cy.quarter_start,
    concat('Q', quarter(cy.quarter_start), ' - ', year(cy.quarter_start)) AS quarter_label,

    -- CY
    cy.sessions,
    cy.visitors,
    cy.bounce_rate,
    cy.quoter_rate,
    cy.bookings,
    cy.transactions,
    cy.gmv_eur,
    cy.nr_eur,
    cy.nr_bt_eur,
    cy.net_tax_eur,
    cy.take_rate_nr,
    cy.take_rate_nr_bt,

    -- PY
    py.sessions        AS py_sessions,
    py.visitors        AS py_visitors,
    py.bounce_rate     AS py_bounce_rate,
    py.quoter_rate     AS py_quoter_rate,
    py.bookings        AS py_bookings,
    py.transactions    AS py_transactions,
    py.gmv_eur         AS py_gmv_eur,
    py.nr_eur          AS py_nr_eur,
    py.nr_bt_eur       AS py_nr_bt_eur,
    py.net_tax_eur     AS py_net_tax_eur,
    py.take_rate_nr    AS py_take_rate_nr,
    py.take_rate_nr_bt AS py_take_rate_nr_bt,

    -- YoY abs
    (cy.sessions     - py.sessions)     AS yoy_sessions_abs,
    (cy.visitors     - py.visitors)     AS yoy_visitors_abs,
    (cy.bookings     - py.bookings)     AS yoy_bookings_abs,
    (cy.transactions - py.transactions) AS yoy_transactions_abs,
    (cy.gmv_eur      - py.gmv_eur)      AS yoy_gmv_eur_abs,
    (cy.nr_eur       - py.nr_eur)       AS yoy_nr_eur_abs,
    (cy.nr_bt_eur    - py.nr_bt_eur)    AS yoy_nr_bt_eur_abs,
    (cy.net_tax_eur  - py.net_tax_eur)  AS yoy_net_tax_eur_abs,

    -- YoY %
    CASE WHEN py.sessions     = 0 THEN NULL ELSE (cy.sessions     / py.sessions)     - 1 END AS yoy_sessions_pct,
    CASE WHEN py.visitors     = 0 THEN NULL ELSE (cy.visitors     / py.visitors)     - 1 END AS yoy_visitors_pct,
    CASE WHEN py.bookings     = 0 THEN NULL ELSE (cy.bookings     / py.bookings)     - 1 END AS yoy_bookings_pct,
    CASE WHEN py.transactions = 0 THEN NULL ELSE (cy.transactions / py.transactions) - 1 END AS yoy_transactions_pct,
    CASE WHEN py.gmv_eur      = 0 THEN NULL ELSE (cy.gmv_eur      / py.gmv_eur)      - 1 END AS yoy_gmv_eur_pct,
    CASE WHEN py.nr_eur       = 0 THEN NULL ELSE (cy.nr_eur       / py.nr_eur)       - 1 END AS yoy_nr_eur_pct,
    CASE WHEN py.nr_bt_eur    = 0 THEN NULL ELSE (cy.nr_bt_eur    / py.nr_bt_eur)    - 1 END AS yoy_nr_bt_eur_pct,
    CASE WHEN py.net_tax_eur  = 0 THEN NULL ELSE (cy.net_tax_eur  / py.net_tax_eur)  - 1 END AS yoy_net_tax_eur_pct
  FROM cy
  LEFT JOIN py
    ON py.quarter_start = add_months(cy.quarter_start, -12)
),

/* -----------------------------
   Long format with formatting:
   - % metrics: *100, round(., 2)
   - others: round(., 0)
   ----------------------------- */
long AS (
  SELECT
    region,
    quarter_label,
    metric,
    value
  FROM final
  LATERAL VIEW stack(
    40,

    -- CY (non-%)
    'cy_sessions',          cast(round(sessions, 0) as double),
    'cy_visitors',          cast(round(visitors, 0) as double),
    'cy_bookings',          cast(round(bookings, 0) as double),
    'cy_transactions',      cast(round(transactions, 0) as double),
    'cy_gmv_eur',           cast(round(gmv_eur, 0) as double),
    'cy_nr_eur',            cast(round(nr_eur, 0) as double),
    'cy_nr_bt_eur',         cast(round(nr_bt_eur, 0) as double),
    'cy_net_tax_eur',       cast(round(net_tax_eur, 0) as double),

    -- CY (%)
    'cy_bounce_rate',       cast(round(bounce_rate * 100, 2) as double),
    'cy_quoter_rate',       cast(round(quoter_rate * 100, 2) as double),
    'cy_take_rate_nr',      cast(round(take_rate_nr * 100, 2) as double),
    'cy_take_rate_nr_bt',   cast(round(take_rate_nr_bt * 100, 2) as double),

    -- PY (non-%)
    'py_sessions',          cast(round(py_sessions, 0) as double),
    'py_visitors',          cast(round(py_visitors, 0) as double),
    'py_bookings',          cast(round(py_bookings, 0) as double),
    'py_transactions',      cast(round(py_transactions, 0) as double),
    'py_gmv_eur',           cast(round(py_gmv_eur, 0) as double),
    'py_nr_eur',            cast(round(py_nr_eur, 0) as double),
    'py_nr_bt_eur',         cast(round(py_nr_bt_eur, 0) as double),
    'py_net_tax_eur',       cast(round(py_net_tax_eur, 0) as double),

    -- PY (%)
    'py_bounce_rate',       cast(round(py_bounce_rate * 100, 2) as double),
    'py_quoter_rate',       cast(round(py_quoter_rate * 100, 2) as double),
    'py_take_rate_nr',      cast(round(py_take_rate_nr * 100, 2) as double),
    'py_take_rate_nr_bt',   cast(round(py_take_rate_nr_bt * 100, 2) as double),

    -- YoY abs (non-%)
    'yoy_sessions_abs',     cast(round(yoy_sessions_abs, 0) as double),
    'yoy_visitors_abs',     cast(round(yoy_visitors_abs, 0) as double),
    'yoy_bookings_abs',     cast(round(yoy_bookings_abs, 0) as double),
    'yoy_transactions_abs', cast(round(yoy_transactions_abs, 0) as double),
    'yoy_gmv_eur_abs',      cast(round(yoy_gmv_eur_abs, 0) as double),
    'yoy_nr_eur_abs',       cast(round(yoy_nr_eur_abs, 0) as double),
    'yoy_nr_bt_eur_abs',    cast(round(yoy_nr_bt_eur_abs, 0) as double),
    'yoy_net_tax_eur_abs',  cast(round(yoy_net_tax_eur_abs, 0) as double),

    -- YoY % (%)
    'yoy_sessions_pct',     cast(round(yoy_sessions_pct * 100, 2) as double),
    'yoy_visitors_pct',     cast(round(yoy_visitors_pct * 100, 2) as double),
    'yoy_bookings_pct',     cast(round(yoy_bookings_pct * 100, 2) as double),
    'yoy_transactions_pct', cast(round(yoy_transactions_pct * 100, 2) as double),
    'yoy_gmv_eur_pct',      cast(round(yoy_gmv_eur_pct * 100, 2) as double),
    'yoy_nr_eur_pct',       cast(round(yoy_nr_eur_pct * 100, 2) as double),
    'yoy_nr_bt_eur_pct',    cast(round(yoy_nr_bt_eur_pct * 100, 2) as double),
    'yoy_net_tax_eur_pct',  cast(round(yoy_net_tax_eur_pct * 100, 2) as double)

  ) s AS metric, value
)

SELECT *
FROM long
PIVOT (
  max(value) FOR quarter_label IN (
    'Q1 - 2026'
  )
)
ORDER BY metric;
